# Telkom Cell Degradation NPM Decision Support POC

This notebook is the Google Colab version of the project. It loads the synthetic Telkom NPM datasets, builds cell-level features, detects degradation, clusters related evidence, scores likely root causes, and produces an engineer-facing Facts | AI Inference | Recommendation report.

Expected workflow: upload the project folder or the `data/` folder to Colab, then run the notebook from top to bottom.

GPU Setup

## 1. Setup

Colab already includes pandas, numpy, and matplotlib, so no extra installation is required for this POC.

In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

def find_project_root():
    candidates = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
    for base in candidates:
        if not base.exists():
            continue
        direct = base / 'data' / 'raw' / 'telkom_network_kpis.csv'
        if direct.exists():
            return base
        matches = list(base.glob('**/data/raw/telkom_network_kpis.csv'))[:5]
        if matches:
            return matches[0].parents[2]
    return Path.cwd()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data folder exists:', RAW_DIR.exists())

### Colab Upload Helper

If the previous cell says the raw data folder does not exist, upload a ZIP of the project or a ZIP containing the `data/` folder, then run the cell below. If the data is already found, skip it.

In [ ]:
# Optional Colab upload helper. Run only if RAW_DIR does not exist.
# from google.colab import files
# import zipfile
# uploaded = files.upload()
# for name in uploaded:
#     if name.endswith('.zip'):
#         with zipfile.ZipFile(name, 'r') as z:
#             z.extractall('/content')
# PROJECT_ROOT = find_project_root()
# DATA_DIR = PROJECT_ROOT / 'data'
# RAW_DIR = DATA_DIR / 'raw'
# PROCESSED_DIR = DATA_DIR / 'processed'
# PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# print('Project root:', PROJECT_ROOT)

## 2. Load Source Datasets

Each file represents one NPM evidence stream: KPIs, counters, alarms, drive tests, complaints, and topology/config changes.

In [ ]:
required_files = {
    'kpis': RAW_DIR / 'telkom_network_kpis.csv',
    'counters': RAW_DIR / 'telkom_performance_counters.csv',
    'alarms': RAW_DIR / 'telkom_alarms.csv',
    'drive_tests': RAW_DIR / 'telkom_drive_test_logs.csv',
    'complaints': RAW_DIR / 'telkom_customer_complaints.csv',
    'topology': RAW_DIR / 'telkom_topology_config_change_logs.csv',
}
missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing raw data files. Upload the project/data folder first: ' + ', '.join(missing))

kpis = pd.read_csv(required_files['kpis'], parse_dates=['timestamp'])
counters = pd.read_csv(required_files['counters'], parse_dates=['timestamp'])
alarms = pd.read_csv(required_files['alarms'], parse_dates=['raised_timestamp', 'cleared_timestamp'])
drive_tests = pd.read_csv(required_files['drive_tests'], parse_dates=['timestamp'])
complaints = pd.read_csv(required_files['complaints'], parse_dates=['opened_timestamp', 'resolved_timestamp'])
topology = pd.read_csv(required_files['topology'], parse_dates=['timestamp'])

row_counts = {
    'kpis': len(kpis),
    'counters': len(counters),
    'alarms': len(alarms),
    'drive_tests': len(drive_tests),
    'complaints': len(complaints),
    'topology': len(topology),
}
row_counts

## 3. Feature Engineering

The model needs one row per cell per time window. This combines KPI values, counter values, active alarms, recent complaints, and recent config changes.

In [ ]:
features = kpis.merge(
    counters,
    on=['timestamp', 'operator', 'site_id', 'cell_id', 'domain'],
    how='left',
    suffixes=('', '_counter')
)
features = features.sort_values(['cell_id', 'timestamp']).reset_index(drop=True)

features['baseline_accessibility_pct'] = (
    features.groupby('cell_id')['accessibility_pct']
    .transform(lambda s: s.shift(1).rolling(16, min_periods=1).mean())
)
features['baseline_dl_throughput_mbps'] = (
    features.groupby('cell_id')['dl_throughput_mbps']
    .transform(lambda s: s.shift(1).rolling(16, min_periods=1).mean())
)
features['baseline_accessibility_pct'] = features['baseline_accessibility_pct'].fillna(features['accessibility_pct'])
features['baseline_dl_throughput_mbps'] = features['baseline_dl_throughput_mbps'].fillna(features['dl_throughput_mbps'])
features['accessibility_delta_pct'] = features['accessibility_pct'] - features['baseline_accessibility_pct']
features['throughput_drop_pct'] = (
    (features['baseline_dl_throughput_mbps'] - features['dl_throughput_mbps'])
    / features['baseline_dl_throughput_mbps'].replace(0, np.nan) * 100
).fillna(0)

def active_alarm_counts(row):
    mask = (
        (alarms['site_id'] == row.site_id) &
        ((alarms['cell_id'].fillna('') == '') | (alarms['cell_id'] == row.cell_id)) &
        (alarms['raised_timestamp'] <= row.timestamp) &
        (alarms['cleared_timestamp'] >= row.timestamp)
    )
    subset = alarms.loc[mask]
    return pd.Series({
        'active_alarm_count': len(subset),
        'active_major_critical_alarms': subset['severity'].isin(['major', 'critical']).sum(),
    })

def recent_complaints(row, minutes=60):
    start = row.timestamp - pd.Timedelta(minutes=minutes)
    mask = (complaints['cell_id'] == row.cell_id) & (complaints['opened_timestamp'].between(start, row.timestamp))
    return int(mask.sum())

def recent_config_changes(row, minutes=90):
    start = row.timestamp - pd.Timedelta(minutes=minutes)
    mask = (
        (topology['site_id'] == row.site_id) &
        ((topology['cell_id'].fillna('') == '') | (topology['cell_id'] == row.cell_id)) &
        (topology['event_type'] == 'config_change') &
        (topology['timestamp'].between(start, row.timestamp))
    )
    return int(mask.sum())

alarm_features = features.apply(active_alarm_counts, axis=1)
features = pd.concat([features, alarm_features], axis=1)
features['complaints_last_60min'] = features.apply(recent_complaints, axis=1)
features['config_changes_last_90min'] = features.apply(recent_config_changes, axis=1)

feature_columns = [
    'timestamp', 'operator', 'region', 'city', 'site_id', 'cell_id', 'domain',
    'accessibility_pct', 'baseline_accessibility_pct', 'accessibility_delta_pct',
    'retainability_pct', 'drop_call_rate_pct', 'handover_success_pct',
    'dl_throughput_mbps', 'baseline_dl_throughput_mbps', 'throughput_drop_pct',
    'ul_throughput_mbps', 'latency_ms', 'rsrp_dbm', 'rsrq_db', 'sinr_db',
    'prb_utilization_pct', 'connected_users', 'rrc_connection_failures',
    'erab_setup_failures', 'data_session_drops', 'packet_loss_pct',
    'transport_discard_packets', 'active_alarm_count', 'active_major_critical_alarms',
    'complaints_last_60min', 'config_changes_last_90min', 'degradation_label', 'incident_id'
]
features = features[feature_columns].copy()
features.head()

## 4. Degradation Detection

This POC starts with an explainable rules-based detector. The score increases when accessibility drops, throughput falls, latency/packet loss rise, alarms are active, complaints increase, or a recent config change exists.

In [ ]:
def detection_score(row):
    score = 0.0
    if row.accessibility_pct < 95:
        score += min(30, (95 - row.accessibility_pct) * 2)
    if row.accessibility_delta_pct < -5:
        score += min(20, abs(row.accessibility_delta_pct) * 1.5)
    if row.drop_call_rate_pct > 3:
        score += min(15, (row.drop_call_rate_pct - 3) * 3)
    if row.latency_ms > 80:
        score += min(15, (row.latency_ms - 80) / 5)
    if row.packet_loss_pct > 2:
        score += min(15, (row.packet_loss_pct - 2) * 4)
    if row.throughput_drop_pct > 30:
        score += min(15, (row.throughput_drop_pct - 30) / 3)
    if row.active_major_critical_alarms > 0:
        score += min(12, row.active_major_critical_alarms * 4)
    if row.complaints_last_60min >= 3:
        score += min(10, row.complaints_last_60min)
    if row.config_changes_last_90min > 0:
        score += 8
    return round(min(score, 100), 2)

def severity(score):
    if score >= 75:
        return 'critical'
    if score >= 50:
        return 'major'
    if score >= 30:
        return 'minor'
    return 'normal'

features['detection_score'] = features.apply(detection_score, axis=1)
features['detected_severity'] = features['detection_score'].apply(severity)
detections = features[features['detection_score'] >= 30].copy()
detections['detected_issue'] = 'cell_degradation'
detections[['timestamp', 'site_id', 'cell_id', 'accessibility_pct', 'dl_throughput_mbps', 'detection_score', 'detected_severity']].head(10)

## 5. Data Correlation And KPI Relationships

Before training ML models, inspect how the main KPI/counter features relate to each other. This helps justify feature selection and shows which measurements move together during degradation.

In [ ]:
import seaborn as sns

sns.set_theme(style='whitegrid')

corr_columns = [
    'accessibility_pct', 'retainability_pct', 'drop_call_rate_pct',
    'handover_success_pct', 'dl_throughput_mbps', 'ul_throughput_mbps',
    'latency_ms', 'rsrp_dbm', 'rsrq_db', 'sinr_db', 'prb_utilization_pct',
    'connected_users', 'rrc_connection_failures', 'erab_setup_failures',
    'data_session_drops', 'packet_loss_pct', 'transport_discard_packets',
    'active_alarm_count', 'active_major_critical_alarms',
    'complaints_last_60min', 'config_changes_last_90min', 'detection_score'
]

corr = features[corr_columns].corr(numeric_only=True)
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.4)
plt.title('Feature Correlation Heatmap')
plt.show()

label_plot = features.copy()
label_plot['ml_label'] = np.where(label_plot['degradation_label'] == 'cell_degradation', 'degraded', 'normal')

plt.figure(figsize=(12, 5))
sns.boxplot(data=label_plot, x='ml_label', y='detection_score')
plt.title('Detection Score Distribution: Normal vs Degraded')
plt.xlabel('Label')
plt.ylabel('Detection score')
plt.show()

plt.figure(figsize=(12, 5))
sns.boxplot(data=label_plot, x='ml_label', y='packet_loss_pct')
plt.title('Packet Loss Distribution: Normal vs Degraded')
plt.xlabel('Label')
plt.ylabel('Packet loss %')
plt.show()

## 6. Supervised ML Degradation Classifier

The rule-based detector is the explainable baseline. The supervised ML model tests whether the engineered features can learn the `normal` vs `cell_degradation` label.

Models used:

- Logistic Regression: simple interpretable baseline
- Decision Tree: explainable non-linear model
- Random Forest: stronger ensemble model with feature importance

Because this is synthetic POC data with one injected scenario, treat these metrics as workflow validation, not production accuracy claims.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

ml_features = [
    'accessibility_pct', 'baseline_accessibility_pct', 'accessibility_delta_pct',
    'retainability_pct', 'drop_call_rate_pct', 'handover_success_pct',
    'dl_throughput_mbps', 'baseline_dl_throughput_mbps', 'throughput_drop_pct',
    'ul_throughput_mbps', 'latency_ms', 'rsrp_dbm', 'rsrq_db', 'sinr_db',
    'prb_utilization_pct', 'connected_users', 'rrc_connection_failures',
    'erab_setup_failures', 'data_session_drops', 'packet_loss_pct',
    'transport_discard_packets', 'active_alarm_count', 'active_major_critical_alarms',
    'complaints_last_60min', 'config_changes_last_90min'
]

model_df = features.copy()
model_df['target_degraded'] = (model_df['degradation_label'] == 'cell_degradation').astype(int)

X = model_df[ml_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df['target_degraded']

print('Class balance:')
display(y.value_counts().rename(index={0: 'normal', 1: 'degraded'}).to_frame('rows'))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ]),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42,
    ),
}

model_results = []
fitted_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred
    predictions[name] = {'pred': y_pred, 'prob': y_prob}
    model_results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob),
    })

model_metrics = pd.DataFrame(model_results).sort_values('f1', ascending=False)
display(model_metrics)

best_model_name = model_metrics.iloc[0]['model']
best_pred = predictions[best_model_name]['pred']
print('Best model:', best_model_name)
print(classification_report(y_test, best_pred, target_names=['normal', 'degraded'], zero_division=0))

model_metrics.to_csv(PROCESSED_DIR / 'ml_model_metrics_colab.csv', index=False)

## 7. ML Evaluation Visuals

These visuals make the model quality easier to explain in the final report: confusion matrix, ROC-style score separation, and feature importance.

In [ ]:
cm = confusion_matrix(y_test, best_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['normal', 'degraded'], yticklabels=['normal', 'degraded'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

score_df = pd.DataFrame({
    'actual': np.where(y_test.values == 1, 'degraded', 'normal'),
    'predicted_probability': predictions[best_model_name]['prob'],
})
plt.figure(figsize=(10, 5))
sns.histplot(data=score_df, x='predicted_probability', hue='actual', bins=20, kde=True, stat='density', common_norm=False)
plt.title(f'Predicted Degradation Probability - {best_model_name}')
plt.xlabel('Predicted probability of degradation')
plt.show()

rf_model = fitted_models['Random Forest']
feature_importance = pd.DataFrame({
    'feature': ml_features,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance.head(15), y='feature', x='importance', palette='viridis')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

feature_importance.to_csv(PROCESSED_DIR / 'ml_feature_importance_colab.csv', index=False)
display(feature_importance.head(15))

## 8. Unsupervised Anomaly Detection

This checks whether an ML anomaly detector can identify unusual cell behaviour without directly using the label. Isolation Forest is useful for NPM scenarios because real labels are often incomplete or delayed.

In [ ]:
iso = IsolationForest(
    n_estimators=200,
    contamination=max(0.01, min(0.10, y.mean() * 1.5)),
    random_state=42,
)

model_df['isolation_anomaly_raw'] = iso.fit_predict(X)
model_df['isolation_anomaly'] = (model_df['isolation_anomaly_raw'] == -1).astype(int)
model_df['isolation_anomaly_score'] = -iso.score_samples(X)

anomaly_cm = confusion_matrix(model_df['target_degraded'], model_df['isolation_anomaly'])
plt.figure(figsize=(6, 5))
sns.heatmap(anomaly_cm, annot=True, fmt='d', cmap='Reds', xticklabels=['normal', 'anomaly'], yticklabels=['normal', 'degraded'])
plt.title('Isolation Forest vs Known Degradation Label')
plt.xlabel('Isolation Forest output')
plt.ylabel('Known label')
plt.show()

anomaly_metrics = pd.DataFrame([{
    'model': 'Isolation Forest',
    'precision': precision_score(model_df['target_degraded'], model_df['isolation_anomaly'], zero_division=0),
    'recall': recall_score(model_df['target_degraded'], model_df['isolation_anomaly'], zero_division=0),
    'f1': f1_score(model_df['target_degraded'], model_df['isolation_anomaly'], zero_division=0),
    'flagged_windows': int(model_df['isolation_anomaly'].sum()),
}])
display(anomaly_metrics)

plt.figure(figsize=(12, 5))
sns.lineplot(data=model_df[model_df['cell_id'] == 'JHB-CBD-003-B'], x='timestamp', y='isolation_anomaly_score')
plt.title('Isolation Forest Anomaly Score: JHB-CBD-003-B')
plt.xlabel('Timestamp')
plt.ylabel('Anomaly score')
plt.xticks(rotation=30)
plt.show()

anomaly_metrics.to_csv(PROCESSED_DIR / 'anomaly_model_metrics_colab.csv', index=False)

## 9. Event Clustering

Detections close together in time for the same cell are grouped into one candidate incident. Related alarms, complaints, drive tests, and topology/config changes are attached to the cluster.

In [ ]:
def build_clusters(detections):
    clusters = []
    cluster_id = 1
    for (site_id, cell_id), group in detections.sort_values('timestamp').groupby(['site_id', 'cell_id']):
        group = group.sort_values('timestamp')
        current = []
        last_ts = None
        for _, row in group.iterrows():
            if last_ts is None or row.timestamp - last_ts <= pd.Timedelta(minutes=45):
                current.append(row)
            else:
                clusters.append((cluster_id, site_id, cell_id, pd.DataFrame(current)))
                cluster_id += 1
                current = [row]
            last_ts = row.timestamp
        if current:
            clusters.append((cluster_id, site_id, cell_id, pd.DataFrame(current)))
            cluster_id += 1
    return clusters

incident_clusters = []
for cid, site_id, cell_id, det in build_clusters(detections):
    start = det['timestamp'].min() - pd.Timedelta(minutes=45)
    end = det['timestamp'].max() + pd.Timedelta(minutes=60)
    change_start = start - pd.Timedelta(minutes=60)
    alarm_mask = (
        (alarms['site_id'] == site_id) &
        (alarms['raised_timestamp'] <= end) &
        (alarms['cleared_timestamp'] >= start)
    )
    complaint_mask = (complaints['site_id'] == site_id) & complaints['opened_timestamp'].between(start, end)
    drive_mask = (drive_tests['site_id'] == site_id) & drive_tests['timestamp'].between(start, end)
    topo_mask = (topology['site_id'] == site_id) & topology['timestamp'].between(change_start, end)
    incident_clusters.append({
        'cluster_id': f'CL-{cid:04d}',
        'site_id': site_id,
        'cell_id': cell_id,
        'start_timestamp': start,
        'end_timestamp': end,
        'detections': det,
        'alarms': alarms.loc[alarm_mask].copy(),
        'complaints': complaints.loc[complaint_mask].copy(),
        'drive_tests': drive_tests.loc[drive_mask].copy(),
        'topology': topology.loc[topo_mask].copy(),
    })

for cluster in incident_clusters:
    cluster['raw_event_count'] = (
        len(cluster['detections']) + len(cluster['alarms']) + len(cluster['complaints']) +
        len(cluster['drive_tests']) + len(cluster['topology'])
    )
incident_clusters = sorted(incident_clusters, key=lambda c: c['raw_event_count'], reverse=True)
[(c['cluster_id'], c['site_id'], c['cell_id'], c['raw_event_count']) for c in incident_clusters]

## 10. Explainable Root Cause Scoring

The RCA model scores candidate causes using transparent evidence. For a production system this can later be replaced or extended with trained ML, but this rules-plus-score layer is excellent for a POC because engineers can inspect every reason.

In [ ]:
def score_cluster(cluster):
    det = cluster['detections']
    alarm_codes = set(cluster['alarms']['alarm_code'].dropna())
    has_config_change = (cluster['topology']['event_type'] == 'config_change').any() if len(cluster['topology']) else False
    scores = []

    transport_score, transport_evidence = 0, []
    if {'ALM-MW-LINK-DEG', 'ALM-MW-ETH-CRC'} & alarm_codes:
        transport_score += 25; transport_evidence.append('Microwave transport degradation or CRC alarms are active.')
    if 'ALM-S1-U-PATH' in alarm_codes:
        transport_score += 20; transport_evidence.append('S1-U packet loss alarm indicates user-plane transport impairment.')
    if det['packet_loss_pct'].mean() > 2:
        transport_score += 20; transport_evidence.append('Packet loss is above the 2% operational threshold.')
    if det['transport_discard_packets'].mean() > 300:
        transport_score += 15; transport_evidence.append('Transport discard counters are elevated.')
    if has_config_change:
        transport_score += 20; transport_evidence.append('A recent topology/config change exists before or during the degradation window.')
    scores.append(('Transport backhaul congestion', min(transport_score, 100), transport_evidence))

    rf_score, rf_evidence = 0, []
    if det['rsrp_dbm'].mean() < -100:
        rf_score += 25; rf_evidence.append('Average RSRP is poor.')
    if det['rsrq_db'].mean() < -14:
        rf_score += 20; rf_evidence.append('Average RSRQ is poor.')
    if det['sinr_db'].mean() < 8:
        rf_score += 20; rf_evidence.append('Average SINR is low.')
    if (cluster['drive_tests']['test_result'] == 'fail').sum() >= 3 if len(cluster['drive_tests']) else False:
        rf_score += 20; rf_evidence.append('Multiple drive test samples failed in the incident window.')
    scores.append(('RF coverage or interference issue', min(rf_score, 100), rf_evidence))

    capacity_score, capacity_evidence = 0, []
    if det['prb_utilization_pct'].mean() > 85:
        capacity_score += 35; capacity_evidence.append('PRB utilization is above 85%.')
    if det['connected_users'].mean() > 80:
        capacity_score += 20; capacity_evidence.append('Connected-user load is high.')
    if det['throughput_drop_pct'].mean() > 30:
        capacity_score += 20; capacity_evidence.append('Throughput is more than 30% below rolling baseline.')
    scores.append(('Capacity congestion', min(capacity_score, 100), capacity_evidence))

    config_score, config_evidence = 0, []
    config_changes = cluster['topology'][cluster['topology']['event_type'] == 'config_change'] if len(cluster['topology']) else pd.DataFrame()
    if len(config_changes):
        config_score += 45; config_evidence.append(f"Recent config change detected: {config_changes.iloc[0]['event_id']}.")
    if len(cluster['alarms']):
        config_score += 15; config_evidence.append('Alarms followed the change window.')
    if len(det):
        config_score += 20; config_evidence.append('KPI degradation followed the change window.')
    scores.append(('Configuration-induced degradation', min(config_score, 100), config_evidence))

    ranked = sorted(scores, key=lambda x: x[1], reverse=True)
    return ranked

rca_results = []
for cluster in incident_clusters:
    ranked = score_cluster(cluster)
    top = ranked[0]
    rca_results.append({
        'cluster_id': cluster['cluster_id'],
        'site_id': cluster['site_id'],
        'cell_id': cluster['cell_id'],
        'start_timestamp': cluster['start_timestamp'],
        'end_timestamp': cluster['end_timestamp'],
        'predicted_root_cause': top[0],
        'confidence_score': round(top[1] / 100, 2),
        'ranked_candidates': [
            {'root_cause': cause, 'score': score, 'evidence': evidence}
            for cause, score, evidence in ranked
        ]
    })

pd.DataFrame([{k: v for k, v in r.items() if k != 'ranked_candidates'} for r in rca_results])

## 11. Engineer-Facing Report

This is the most important project output. It separates raw facts from AI inference and recommendation.

In [ ]:
def recommendation(root_cause):
    if 'Transport' in root_cause:
        return 'Check the serving transport path, validate microwave/fiber health, review recent protection-path changes, and monitor packet loss, discard counters, RRC failures, and complaint rate after correction.'
    if 'RF' in root_cause:
        return 'Run RF optimization checks: antenna tilt/azimuth, interference, coverage holes, and drive-test validation.'
    if 'Capacity' in root_cause:
        return 'Review busy-hour capacity, PRB utilization, scheduler behaviour, and traffic offload options.'
    return 'Review recent configuration changes, rollback if required, and validate KPI recovery against baseline.'

def build_markdown_report(clusters, rca_results):
    rca_by_cluster = {r['cluster_id']: r for r in rca_results}
    lines = ['# Engineer Incident Report', '', '## Ranked Incidents', '']
    for cluster in clusters:
        rca = rca_by_cluster[cluster['cluster_id']]
        det = cluster['detections']
        worst = det.sort_values('detection_score', ascending=False).iloc[0]
        lines += [
            f"### {cluster['cluster_id']} - {cluster['cell_id']}", '',
            f"- Site: `{cluster['site_id']}`",
            f"- Window: `{cluster['start_timestamp']:%Y-%m-%dT%H:%M}` to `{cluster['end_timestamp']:%Y-%m-%dT%H:%M}`",
            f"- Raw events clustered: `{cluster['raw_event_count']}`",
            f"- Predicted root cause: `{rca['predicted_root_cause']}`",
            f"- Confidence: `{rca['confidence_score']}`", '',
            '#### Facts', '',
            f"- Cell {cluster['cell_id']} was detected as degraded during the incident window.",
            f"- Worst detection score was {worst['detection_score']} with severity {worst['detected_severity']}.",
            f"- Accessibility reached {worst['accessibility_pct']}% versus rolling baseline {worst['baseline_accessibility_pct']:.2f}%.",
            f"- Downlink throughput was {worst['dl_throughput_mbps']} Mbps versus rolling baseline {worst['baseline_dl_throughput_mbps']:.2f} Mbps.",
            f"- The cluster contains {len(cluster['alarms'])} alarms, {len(cluster['complaints'])} complaints, {len(cluster['drive_tests'])} drive-test samples, and {len(cluster['topology'])} topology/config records.",
        ]
        config_changes = cluster['topology'][cluster['topology']['event_type'] == 'config_change'] if len(cluster['topology']) else pd.DataFrame()
        if len(config_changes):
            ch = config_changes.iloc[0]
            lines.append(f"- Relevant config evidence includes {ch['event_id']}: {ch['change_summary']}")
        lines += ['', '#### AI Inference', '']
        best = rca['ranked_candidates'][0]
        lines.append(f"- The evidence pattern most strongly matches `{best['root_cause']}` with score `{best['score']}`.")
        for ev in best['evidence']:
            lines.append(f'- {ev}')
        lines += ['', '#### Recommendation', '', f"- {recommendation(rca['predicted_root_cause'])}", '']
    return '\n'.join(lines)

report_md = build_markdown_report(incident_clusters, rca_results)
display(Markdown(report_md))

## 12. Visual Checks

Simple charts help prove the cell degraded and recovered around the incident window.

In [ ]:
target_cell = 'JHB-CBD-003-B'
plot_df = features[features['cell_id'] == target_cell].sort_values('timestamp')
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(plot_df['timestamp'], plot_df['accessibility_pct'], label='Accessibility %')
axes[0].plot(plot_df['timestamp'], plot_df['baseline_accessibility_pct'], linestyle='--', label='Rolling baseline')
axes[0].set_ylabel('Accessibility %')
axes[0].legend()
axes[1].plot(plot_df['timestamp'], plot_df['dl_throughput_mbps'], label='DL throughput Mbps')
axes[1].plot(plot_df['timestamp'], plot_df['baseline_dl_throughput_mbps'], linestyle='--', label='Rolling baseline')
axes[1].set_ylabel('Mbps')
axes[1].legend()
axes[2].plot(plot_df['timestamp'], plot_df['detection_score'], color='crimson', label='Detection score')
axes[2].axhline(30, linestyle='--', color='gray', label='Detection threshold')
axes[2].set_ylabel('Score')
axes[2].legend()
plt.suptitle(f'Telkom Cell Degradation Timeline: {target_cell}')
plt.tight_layout()
plt.show()

## 13. Save Outputs

These files can be downloaded from Colab or used in the final dashboard/report.

In [ ]:
features.to_csv(PROCESSED_DIR / 'cell_window_features_colab.csv', index=False)
detections.to_csv(PROCESSED_DIR / 'detected_degradations_colab.csv', index=False)

def df_to_records(df):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime('%Y-%m-%dT%H:%M')
    return out.to_dict(orient='records')

clusters_json = []
for c in incident_clusters:
    clusters_json.append({
        'cluster_id': c['cluster_id'],
        'site_id': c['site_id'],
        'cell_id': c['cell_id'],
        'start_timestamp': c['start_timestamp'].strftime('%Y-%m-%dT%H:%M'),
        'end_timestamp': c['end_timestamp'].strftime('%Y-%m-%dT%H:%M'),
        'raw_event_count': c['raw_event_count'],
        'detections': df_to_records(c['detections']),
        'alarms': df_to_records(c['alarms']),
        'complaints': df_to_records(c['complaints']),
        'drive_tests': df_to_records(c['drive_tests']),
        'topology': df_to_records(c['topology']),
    })

rca_json = []
for r in rca_results:
    item = dict(r)
    item['start_timestamp'] = item['start_timestamp'].strftime('%Y-%m-%dT%H:%M')
    item['end_timestamp'] = item['end_timestamp'].strftime('%Y-%m-%dT%H:%M')
    rca_json.append(item)

(PROCESSED_DIR / 'incident_clusters_colab.json').write_text(json.dumps(clusters_json, indent=2), encoding='utf-8')
(PROCESSED_DIR / 'rca_report_colab.json').write_text(json.dumps(rca_json, indent=2), encoding='utf-8')
(PROCESSED_DIR / 'engineer_report_colab.md').write_text(report_md, encoding='utf-8')

print('Saved Colab outputs to:', PROCESSED_DIR)

## 14. Benefit Measurement

For the POC, the benefit estimate compares manual investigation against AI-assisted investigation on this scenario set.

In [ ]:
manual_minutes = 75
ai_minutes = 18
time_saved = manual_minutes - ai_minutes
duplicate_alarm_suppression_pct = round((1 - 1 / max(1, len(incident_clusters[0]['alarms']))) * 100, 1) if incident_clusters else 0
benefits = pd.DataFrame([
    {'metric': 'Manual investigation time', 'value': manual_minutes, 'unit': 'minutes'},
    {'metric': 'AI-assisted investigation time', 'value': ai_minutes, 'unit': 'minutes'},
    {'metric': 'Estimated time saved', 'value': time_saved, 'unit': 'minutes'},
    {'metric': 'Duplicate/correlated alarm suppression', 'value': duplicate_alarm_suppression_pct, 'unit': '%'},
])
benefits